In [2]:
#####################################################
# DIAGNÓSTICO 2: CRUCE ENTRE HOJAS
# Marcas, nombres de producto, estructura de salidas y fechas de ventas
#####################################################
#Importamos librerias
import pandas as pd

#####################################################
#DATOS A LLENAR (solo modificar esta sección)
ARCHIVO = "Datos_CLV.xlsx"
HOJA_EXIST = "Existencias_totales"
HOJA_SALIDAS = "Histórico_de_Entradas_Salidas"
HOJA_VENTAS = "Histórico_Ventas"
#####################################################

#Lectura de las hojas
exist = pd.read_excel(ARCHIVO, sheet_name=HOJA_EXIST)
salidas = pd.read_excel(ARCHIVO, sheet_name=HOJA_SALIDAS)
ventas = pd.read_excel(ARCHIVO, sheet_name=HOJA_VENTAS)

#####################################################
#Función para normalizar texto (mayúsculas y solo letras y números)
def normalizar(serie):
    return (serie.astype(str)
                 .str.upper()
                 .str.strip()
                 .str.replace(r"[^A-Z0-9]", "", regex=True))

#####################################################
#1. MARCAS DE CADA HOJA (para unificar nombres)
print("=" * 60)
print("1. MARCAS POR HOJA (con número de filas)")
print("=" * 60)
print("\nExistencias:")
print(exist["marca_proveniente "].value_counts().to_string())
print("\nSalidas:")
print(salidas["MARCA_PROVENIENTE"].value_counts().to_string())
print("\nVentas:")
print(ventas["marca_"].value_counts().to_string())

#Marcas que no coinciden después de normalizar
m_exist = set(normalizar(exist["marca_proveniente "]))
m_sal = set(normalizar(salidas["MARCA_PROVENIENTE"]))
m_ven = set(normalizar(ventas["marca_"]))
print("\nMarcas normalizadas en común (las 3 hojas):", sorted(m_exist & m_sal & m_ven))
print("Solo en existencias:", sorted(m_exist - m_sal - m_ven))
print("Solo en salidas:", sorted(m_sal - m_exist - m_ven))
print("Solo en ventas:", sorted(m_ven - m_exist - m_sal))

#####################################################
#2. CRUCE POR NOMBRE DE PRODUCTO
print("\n" + "=" * 60)
print("2. CRUCE POR NOMBRE DE PRODUCTO")
print("=" * 60)
n_exist = set(normalizar(exist["nombre_articulos"]))
n_sal = set(normalizar(salidas["NOMBRE_PRODUCTO"]))
n_ven = set(normalizar(ventas["nombre_articulo_en_existencia"]))
print(f"Existencias ({len(n_exist)}) vs Salidas ({len(n_sal)}): {len(n_exist & n_sal)} en común")
print(f"Ventas ({len(n_ven)}) vs Salidas ({len(n_sal)}): {len(n_ven & n_sal)} en común")
print(f"Existencias ({len(n_exist)}) vs Ventas ({len(n_ven)}): {len(n_exist & n_ven)} en común")

#Ejemplos de nombres de cada hoja para comparar el formato
print("\nEjemplos de nombres:")
print("Existencias ->", exist["nombre_articulos"].head(5).tolist())
print("Salidas     ->", salidas["NOMBRE_PRODUCTO"].head(5).tolist())
print("Ventas      ->", ventas["nombre_articulo_en_existencia"].head(5).tolist())

#¿Algún SKU coincide con articulo_id? (por si el código es el mismo)
sku = set(salidas["SKU_INVENTARIO"].astype(str).str.strip())
ids = set(exist["articulo_id"].astype(str))
print("\nSKU de salidas que coinciden con articulo_id de existencias:", len(sku & ids))

#####################################################
#3. ESTRUCTURA DE SALIDAS
print("\n" + "=" * 60)
print("3. ESTRUCTURA DE SALIDAS")
print("=" * 60)
print("\nFilas por año:")
print(salidas["ANO_PROVENIENTE"].value_counts().sort_index().to_string())
print("\nMeses con datos por año:")
print(salidas.groupby("ANO_PROVENIENTE")["MES_ANO"].nunique().to_string())
print("\nSalidas totales por año:")
print(salidas.groupby("ANO_PROVENIENTE")["SALIDAS_"].sum().to_string())
print("\nÚltimo mes con datos en 2026:", salidas.loc[salidas["ANO_PROVENIENTE"] == 2026, "MES_ANO"].max())

#¿Cada SKU aparece una sola vez por año y mes?
repetidos = salidas.duplicated(subset=["ANO_PROVENIENTE", "MES_ANO", "SKU_INVENTARIO"]).sum()
print("\nFilas repetidas por (año, mes, SKU):", repetidos)

#####################################################
#4. FECHAS DE VENTAS Y VALORES EN CERO
print("\n" + "=" * 60)
print("4. FECHAS DE VENTAS Y CEROS")
print("=" * 60)
fechas = pd.to_datetime(ventas["fecha_documento_PROVENIENTE"], format="%d.%m.%Y", errors="coerce")
print("Rango real de fechas:", fechas.min().date(), "->", fechas.max().date())
print("Fechas que no se pudieron convertir:", fechas.isnull().sum())
print("\nPartidas por mes:")
print(fechas.dt.month.value_counts().sort_index().to_string())

print("\nArtículos en existencias con valor de inventario en 0:",
      (exist["valor_inventario_articulo"] == 0).sum())
print("Partidas de ventas con precio 0:", (ventas["precio_unitari_o"] == 0).sum())

1. MARCAS POR HOJA (con número de filas)

Existencias:
marca_proveniente 
Vita              1810
Amann Girrbach     310
Creation           241
Aidite             215
Anax Dent           97
Huge                88
Jensen              42
Scheftner           34
Phrozen             24
Asiga               12
Hass                11
Medit                9
Roko                 8
Renfert              7
Dekema               6
Dess                 6
Schick               6
Exocad               5
Dentine              4
Quatro               4
3dgar                3
Durr Dental          2
Cyberpuerta          2
Parmex               2
Arma Dental          2
Apple                1

Salidas:
MARCA_PROVENIENTE
VITA                              29373
AG                                 7458
AIDITE                             3255
Komet                              2574
HUGE                               2044
JENSE                              1826
YETI                               1730
RENFERT             